In [1]:
import os
import yaml

YAML_PATH = "/kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/merged_data.yaml"
BASE_PATH = "/kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis"

# Check YAML
print("📋 YAML contents:")
with open(YAML_PATH, 'r') as f:
    print(f.read())

# Count images
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(BASE_PATH, split, 'images')
    lbl_dir = os.path.join(BASE_PATH, split, 'labels')
    imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"{split}: {imgs} images, {lbls} labels")

📋 YAML contents:
names:
- fracture
- no_fracture
nc: 2
path: .
test: test/images
train: train/images
val: valid/images

train: 11840 images, 11840 labels
valid: 184 images, 184 labels
test: 194 images, 194 labels


# **BLOCK 1: INSTALL YOLOv8 AND DEPENDENCIES**

In [ ]:
# ============================================================================
# BLOCK 1: INSTALL YOLOv8 AND DEPENDENCIES
# ============================================================================

print("="*70)
print("INSTALLING YOLOv8 FOR HIPS & PELVIS FRACTURE DETECTION")
print("="*70)

!pip install ultralytics -q

import ultralytics
from ultralytics import YOLO
import torch
import os
import yaml
import shutil
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

print(f"✅ YOLOv8 version: {ultralytics.__version__}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*70)

# **BLOCK 2: DATASET PATHS AND VERIFICATION**

In [ ]:
# ============================================================================
# BLOCK 2: VERIFY DATASET PATHS
# ============================================================================

print("="*70)
print("HIPS & PELVIS DATASET VERIFICATION")
print("="*70)

# Paths to your dataset on Kaggle
BASE_PATH = "/kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis"

# Paths to splits
TRAIN_IMAGES = os.path.join(BASE_PATH, "train", "images")
TRAIN_LABELS = os.path.join(BASE_PATH, "train", "labels")
VAL_IMAGES = os.path.join(BASE_PATH, "valid", "images")
VAL_LABELS = os.path.join(BASE_PATH, "valid", "labels")
TEST_IMAGES = os.path.join(BASE_PATH, "test", "images")
TEST_LABELS = os.path.join(BASE_PATH, "test", "labels")

# Path to merged YAML
YAML_PATH = "/kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/merged_data.yaml"

WORKING_DIR = "/kaggle/working"

print(f"📁 Base path: {BASE_PATH}")
print(f"📁 Train images: {TRAIN_IMAGES}")
print(f"📁 Val images: {VAL_IMAGES}")
print(f"📁 Test images: {TEST_IMAGES}")
print(f"📁 YAML path: {YAML_PATH}")

# Verify paths exist
for path, name in [(TRAIN_IMAGES, "Train images"), (TRAIN_LABELS, "Train labels"),
                   (VAL_IMAGES, "Val images"), (VAL_LABELS, "Val labels"),
                   (TEST_IMAGES, "Test images"), (TEST_LABELS, "Test labels")]:
    exists = os.path.exists(path)
    print(f"  {name}: {'✅' if exists else '❌'}")

# Count images
train_count = len([f for f in os.listdir(TRAIN_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(TRAIN_IMAGES) else 0
val_count = len([f for f in os.listdir(VAL_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(VAL_IMAGES) else 0
test_count = len([f for f in os.listdir(TEST_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(TEST_IMAGES) else 0

print(f"\n📊 DATASET STATISTICS:")
print(f"   Training: {train_count} images")
print(f"   Validation: {val_count} images")
print(f"   Test: {test_count} images")
print(f"   TOTAL: {train_count + val_count + test_count} images")

# Verify YAML exists
if os.path.exists(YAML_PATH):
    print(f"\n✅ Merged YAML found!")
    with open(YAML_PATH, 'r') as f:
        print(f"\n📋 YAML contents:")
        print("-"*40)
        print(f.read())
        print("-"*40)
else:
    print(f"\n❌ YAML not found at: {YAML_PATH}")

print("="*70)

# **BLOCK 3: COPY MERGED YAML TO WORKING DIRECTORY**

In [ ]:
# ============================================================================
# BLOCK 3: COPY MERGED YAML TO WORKING DIRECTORY
# ============================================================================

print("="*70)
print("COPYING MERGED YAML TO WORKING DIRECTORY")
print("="*70)

# Copy the merged YAML to working directory
local_yaml_path = os.path.join(WORKING_DIR, "data.yaml")

if os.path.exists(YAML_PATH):
    shutil.copy(YAML_PATH, local_yaml_path)
    print(f"✅ Copied YAML to: {local_yaml_path}")
    
    # Update the path in YAML to point to the correct location
    with open(local_yaml_path, 'r') as f:
        yaml_content = yaml.safe_load(f)
    
    # Update the path to the actual dataset location
    yaml_content['path'] = BASE_PATH
    
    with open(local_yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    
    print(f"✅ Updated YAML with correct path: {BASE_PATH}")
    print("\n📋 Updated YAML contents:")
    with open(local_yaml_path, 'r') as f:
        print(f.read())
else:
    print(f"❌ YAML not found, creating one...")
    # Create YAML if not found
    data_yaml = {
        'path': BASE_PATH,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 2,
        'names': ['fracture', 'no_fracture']
    }
    with open(local_yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    print(f"✅ Created YAML at: {local_yaml_path}")

print("="*70)

# **BLOCK 4: TRAIN YOLOv8 ON HIPS & PELVIS FRACTURE DATASET**

In [ ]:
# ============================================================================
# BLOCK 4: TRAIN YOLOv8 ON HIPS & PELVIS FRACTURE DATASET
# ============================================================================

print("="*70)
print("TRAINING YOLOv8 ON HIPS & PELVIS FRACTURE DATASET")
print("="*70)
print(f"📊 Training images: {train_count}")
print(f"📊 Validation images: {val_count}")
print(f"📊 Test images: {test_count}")
print(f"🎯 Target epochs: 100")
print(f"⏱️ Estimated time: 8-12 hours")
print(f"💾 Batch size: 16")
print(f"🖼️ Image size: 640")
print(f"📋 Classes: fracture (0), no_fracture (1)")
print("="*70)

# Initialize YOLOv8 with pretrained weights
model = YOLO('yolov8m.pt')

# Training parameters
results = model.train(
    data=local_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=8,
    
    # Save settings
    save=True,
    save_period=10,
    project='hips_pelvis_yolo',
    name='run1_100epochs',
    exist_ok=True,
    
    # Early stopping
    patience=20,
    
    # Optimizer settings
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    
    # Data augmentation
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
)

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

# **BLOCK 5: EVALUATE ON VALIDATION SET**

In [ ]:
# ============================================================================
# BLOCK 5: EVALUATE ON VALIDATION SET
# ============================================================================

print("="*70)
print("EVALUATING ON VALIDATION SET")
print("="*70)

# Load best model from training
best_model_path = '/kaggle/working/hips_pelvis_yolo/run1_100epochs/weights/best.pt'
model = YOLO(best_model_path)

# Run validation
val_results = model.val(data=local_yaml_path, split='val')

print("\n📊 VALIDATION RESULTS:")
print("-"*50)
print(f"   mAP50 (IoU=0.50): {val_results.box.map50:.2f}%")
print(f"   mAP50-95 (IoU=0.50:0.95): {val_results.box.map:.2f}%")
print(f"   Precision: {val_results.box.mp:.2f}%")
print(f"   Recall: {val_results.box.mr:.2f}%")
print("-"*50)
print("="*70)

# **BLOCK 6: EVALUATE ON TEST SET**

In [ ]:
# ============================================================================
# BLOCK 6: EVALUATE ON TEST SET
# ============================================================================

print("="*70)
print("EVALUATING ON TEST SET")
print("="*70)

# Run test evaluation
test_results = model.val(data=local_yaml_path, split='test')

print("\n📊 TEST RESULTS:")
print("-"*50)
print(f"   mAP50 (IoU=0.50): {test_results.box.map50:.2f}%")
print(f"   mAP50-95 (IoU=0.50:0.95): {test_results.box.map:.2f}%")
print(f"   Precision: {test_results.box.mp:.2f}%")
print(f"   Recall: {test_results.box.mr:.2f}%")
print("-"*50)
print("="*70)

# **BLOCK 9: SAVE BEST MODEL FOR DOWNLOAD**

In [ ]:
# ============================================================================
# BLOCK 9: SAVE BEST MODEL FOR DOWNLOAD
# ============================================================================

print("="*70)
print("SAVING BEST MODEL FOR DOWNLOAD")
print("="*70)

# Copy best model to working directory
best_model_source = '/kaggle/working/shoulder_arm_yolo/run1_100epochs/weights/best.pt'
best_model_dest = '/kaggle/working/yolo_best_model.pt'
last_model_dest = '/kaggle/working/yolo_last_model.pt'

if os.path.exists(best_model_source):
    shutil.copy(best_model_source, best_model_dest)
    print(f"✅ Best model saved to: {best_model_dest}")
    
    # Also copy last model
    last_model_source = '/kaggle/working/shoulder_arm_yolo/run1_100epochs/weights/last.pt'
    if os.path.exists(last_model_source):
        shutil.copy(last_model_source, last_model_dest)
        print(f"✅ Last model saved to: {last_model_dest}")
else:
    print(f"❌ Model not found at: {best_model_source}")

# Create downloadable link
from IPython.display import FileLink, display

print("\n📥 Download links:")
if os.path.exists(best_model_dest):
    display(FileLink(best_model_dest))
if os.path.exists(last_model_dest):
    display(FileLink(last_model_dest))

# Save results to JSON
import json
results_summary = {
    'train_images': train_count,
    'val_images': val_count,
    'test_images': test_count,
    'epochs': 100,
    'val_mAP50': float(val_results.box.map50),
    'val_mAP50_95': float(val_results.box.map),
    'test_mAP50': float(test_results.box.map50),
    'test_mAP50_95': float(test_results.box.map),
    'precision': float(test_results.box.mp),
    'recall': float(test_results.box.mr)
}

with open('/kaggle/working/training_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("\n✅ Results saved to: /kaggle/working/training_results.json")
print("="*70)